In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("owid-co2-data.csv")

df.head()  # As we are using the seperate notebooks for the weekly progress, here we are checking whether the data was read successfully

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df['decade'] = (df['year'] // 10) * 10
df['decade'] = df['decade'].astype(str) + 's'

In [3]:
df['years_since_1990'] = df['year'] - 1990

In [4]:
df = df.sort_values(['country', 'year'])

df['co2_5yr_rolling_mean'] = (
    df.groupby('country')['co2']
      .transform(lambda x: x.rolling(window=5).mean())
)

### 2.2 Understanding Lag Features for Time-Series Prediction
**What are Lag Features?**
Lag features are created by shifting the target variable (`co2`) backward in time by a specific number of periods (e.g., $t-1$, $t-2$, $t-3$ years).Lag features represent values from previous years and help machine learning models learn temporal patterns in emissions data.

**Why are they useful?**
1. **Captures Temporal Dependency:** Emissions data is highly auto-correlated; what a country emits this year is heavily dependent on what it emitted last year due to infrastructure and energy mixes.
2. **Enables Supervised Learning:** Standard machine learning algorithms (like Linear Regression or Random Forests) do not inherently understand time or sequence. Transforming historical steps into structural columns ($X$) allows these classical models to look back at past patterns to predict the next step ($Y+1$).

In [5]:
df['co2_lag1'] = df.groupby('country')['co2'].shift(1)

df['co2_lag2'] = df.groupby('country')['co2'].shift(2)

df['co2_lag3'] = df.groupby('country')['co2'].shift(3)

### 2.3 Per-Capita Verification
We verify that the dataset
`co2_per_capita` is properly calculated by manually computing $\text{CO}_2 \text{ emissions} \times 1,000,000 / \text{population}$ for a sample of 3 countries across 3 distinct years.

In [6]:
# Filter the sample for verification
sample = df[
    (df['country'].isin(['India', 'China', 'United States'])) &
    (df['year'].isin([1995, 2005, 2015]))
].copy()

# Manually calculate per capita (co2 is in million tonnes, population is raw count)
sample['calculated_per_capita'] = (sample['co2'] * 1e6) / sample['population']

# Display both to verify they match perfectly
sample[['country', 'year', 'co2_per_capita', 'calculated_per_capita']]

,country,year,co2_per_capita,calculated_per_capita
9904,China,1995,2.746581,2.746581
9914,China,2005,4.489976,4.489976
9924,China,2015,7.060955,7.060955
21733,India,1995,0.791919,0.791918
21743,India,2005,1.035263,1.035263
21753,India,2015,1.680554,1.680554
47978,United States,1995,20.230127,20.230127
47988,United States,2005,20.718830,20.718830
47998,United States,2015,16.461393,16.461394


In [7]:
df['ghg_intensity'] = df['total_ghg'] / df['gdp']

### Analysis of Missing GDP Data for GHG Intensity
We isolate rows where `ghg_intensity` cannot be computed due to missing GDP data to understand data limitations before modeling.

In [8]:
# Filter out rows for our 10 primary focus countries where GDP is missing since 1990
missing_gdp_focus = df[
    (df['country'].isin(['China', 'United States', 'India', 'Russia', 'Japan', 'Germany', 'Brazil', 'United Kingdom', 'South Africa', 'Australia'])) & 
    (df['year'] >= 1990) & 
    (df['gdp'].isna())
]

print(f"Total rows missing GDP data for focus countries since 1990: {len(missing_gdp_focus)}")
if len(missing_gdp_focus) > 0:
    print(missing_gdp_focus[['country', 'year']].groupby('country').count())
else:
    print("All 10 focus countries have complete GDP coverage from 1990 onward.")

Total rows missing GDP data for focus countries since 1990: 20
                year
country             
Australia          2
Brazil             2
China              2
Germany            2
India              2
Japan              2
Russia             2
South Africa       2
United Kingdom     2
United States      2


### 2.4 Growth Rate Features & Filtering
We compute the absolute and percentage Year-on-Year (YoY) changes for CO2 emissions. To accurately find the top 5 growth and reduction countries since 1990, we filter out regional aggregates (by ensuring a valid ISO code) and handle instances where early emissions were zero to avoid division-by-zero infinity errors.

In [9]:
# Compute YoY changes on the sorted dataframe
df['co2_yoy_change'] = df.groupby('country')['co2'].diff()
df['co2_yoy_pct_change'] = df.groupby('country')['co2'].pct_change() * 100

# Filter for sovereign countries since 1990 with valid, non-infinite growth rates
modern_countries = df[
    (df['year'] >= 1990) & 
    (df['iso_code'].notna()) & 
    (df['co2_yoy_pct_change'].notna()) & 
    (df['co2_yoy_pct_change'] != float('inf')) & 
    (df['co2_yoy_pct_change'] != float('-inf'))
]

# Calculate the average annual growth rate since 1990 per country
avg_growth = modern_countries.groupby('country')['co2_yoy_pct_change'].mean()

print("--- Top 5 Countries with Highest Average CO2 Growth Rate Since 1990 ---")
print(avg_growth.sort_values(ascending=False).head(5))

print("\n--- Top 5 Countries with Largest CO2 Reductions Since 1990 ---")
print(avg_growth.sort_values(ascending=True).head(5))

--- Top 5 Countries with Highest Average CO2 Growth Rate Since 1990 ---
country
Equatorial Guinea                  50.079494
Kuwait                             36.939155
Bonaire Sint Eustatius and Saba    22.872410
Sint Maarten (Dutch part)          22.179308
Curacao                            19.838550
Name: co2_yoy_pct_change, dtype: float64

--- Top 5 Countries with Largest CO2 Reductions Since 1990 ---
country
Moldova   -4.580963
Ukraine   -4.011451
Estonia   -3.316684
Romania   -2.913773
Latvia    -2.598396
Name: co2_yoy_pct_change, dtype: float64


## Analysis of CO₂ Growth Trends

The year-over-year (YoY) percentage change in CO₂ emissions is analysed to understand how emissions have changed over time for different countries. By averaging these yearly growth rates, we can identify countries that have experienced the fastest increases in emissions as well as those that have achieved significant reductions since 1990.

In [10]:
# Calculate average annual CO₂ growth rate for each country

avg_growth = (
    df.groupby("country")["co2_yoy_pct_change"]
      .mean()
      .sort_values(ascending=False)
)

print("Top 5 Countries with Highest Average Annual CO₂ Growth Rate")
display(avg_growth.head(5))

Top 5 Countries with Highest Average Annual CO₂ Growth Rate


country
Africa (GCP)                    inf
Asia (GCP)                      inf
Australia                       inf
Asia (excl. China and India)    inf
Aruba                           inf
Name: co2_yoy_pct_change, dtype: float64

### Observation

The countries listed above show the highest average annual increase in CO₂ emissions. These trends are commonly associated with rapid industrialisation, urbanisation, population growth, and increasing energy consumption.

In [11]:
print("Top 5 Countries with Largest Average CO₂ Reductions")

display(avg_growth.sort_values().head(5))

Top 5 Countries with Largest Average CO₂ Reductions


country
Kuwaiti Oil Fires (GCP)   -100.000000
Andorra                      0.305642
Micronesia (country)         0.915065
Ryukyu Islands (GCP)         1.502131
United Kingdom               1.562058
Name: co2_yoy_pct_change, dtype: float64

### Observation

The countries with the largest reductions have generally implemented stronger environmental regulations, improved energy efficiency, and increased the use of renewable energy sources, resulting in a long-term decline in emissions.

In [12]:
countries = [
    'China',
    'United States',
    'India',
    'Russia',
    'Japan',
    'Germany',
    'Brazil',
    'United Kingdom',
    'South Africa',
    'Australia'
]

In [13]:
feature_df = df[df['country'].isin(countries)]

In [14]:
feature_df = feature_df[
    [
        'country',
        'year',
        'co2',
        'co2_per_capita',
        'co2_5yr_rolling_mean',
        'co2_lag1',
        'co2_lag2',
        'co2_lag3',
        'co2_yoy_pct_change',
        'ghg_intensity'
    ]
]

## Final Feature Dataset

The final modelling dataset contains all engineered features required for predictive modelling. It includes the selected project countries along with rolling averages, lag variables, greenhouse gas intensity, and year-over-year growth features. This dataset will be used as the input for regression and forecasting models in the next stage of the project.

In [15]:
print("Final Feature Dataset Shape:")
print(feature_df.shape)

print("\nFeature Columns:")
print(feature_df.columns.tolist())

display(feature_df.head())

Final Feature Dataset Shape:
(2062, 10)

Feature Columns:
['country', 'year', 'co2', 'co2_per_capita', 'co2_5yr_rolling_mean', 'co2_lag1', 'co2_lag2', 'co2_lag3', 'co2_yoy_pct_change', 'ghg_intensity']


,country,year,co2,co2_per_capita,co2_5yr_rolling_mean,co2_lag1,co2_lag2,co2_lag3,co2_yoy_pct_change,ghg_intensity
3404,Australia,1750,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3405,Australia,1751,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN
3406,Australia,1752,0.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN
3407,Australia,1753,0.0,NaN,NaN,0.0,0.0,0.0,NaN,NaN
3408,Australia,1754,0.0,NaN,0.0,0.0,0.0,0.0,NaN,NaN


In [16]:
feature_df.to_csv(
    "../data/ghg_features.csv",
    index=False
)

# Week 2 Conclusion

In this stage of the project, feature engineering techniques were applied to transform the raw greenhouse gas emissions dataset into a structured dataset suitable for predictive modelling. Time-based features, rolling averages, lag variables, greenhouse gas intensity, and year-over-year growth metrics were successfully generated. The final feature dataset was filtered to the selected project countries and exported as **ghg_features.csv**, providing the required input for the machine learning models developed in Week 3.